# Baseline: Basic RAG with LangGraph

This notebook is the starting point of my project. It is the simplest RAG pipeline: **retrieve the top-k chunks, then generate an answer**. I built it first so I have a baseline to compare Corrective RAG against (`02_corrective_rag.ipynb`).

Its weakness is that it never checks whether the retrieved chunks are actually relevant. It trusts whatever the retriever returns.

## 0. Install dependencies
Installs the libraries into the same Python environment the notebook is running in. Needs to be run only once.

In [1]:
%pip install -U python-dotenv pypdf faiss-cpu langchain-core langchain-community langchain-text-splitters langchain-google-genai langchain-groq langchain-tavily langgraph

   ---------------------------------------- 0.0/405.1 kB ? eta -:--:--
   ---- ----------------------------------- 41.0/405.1 kB ? eta -:--:--
   ----------- ---------------------------- 112.6/405.1 kB 1.6 MB/s eta 0:00:01
   ----------- ---------------------------- 112.6/405.1 kB 1.6 MB/s eta 0:00:01
   ----------- ---------------------------- 112.6/405.1 kB 1.6 MB/s eta 0:00:01
   ------------------------ --------------- 245.8/405.1 kB 1.3 MB/s eta 0:00:01
   ------------------------ --------------- 245.8/405.1 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 405.1/405.1 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/573.8 kB ? eta -:--:--
   --------------------------- ----------- 409.6/573.8 kB 12.9 MB/s eta 0:00:01
   ------------------------------------- - 553.0/573.8 kB 11.5 MB/s eta 0:00:01
   ------------------------------------- - 553.0/573.8 kB 11.5 MB/s eta 0:00:01
   ------------------------------------- - 553.0/573.8 kB 11.5 MB/


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports
LangChain for loading, splitting and prompting, FAISS as the vector store, Gemini for embeddings, Groq for the LLM, and LangGraph to connect the steps into a graph.

In [2]:
import re
from pathlib import Path
from typing import List, TypedDict

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph

C:\Users\MAHEK\AppData\Local\Temp\ipykernel_10812\2742675339.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. Configuration
API keys are loaded from a `.env` file so they never appear in the code. `TOP_K = 4` means the retriever returns the 4 most similar chunks: enough context to answer, but few enough to keep the prompt short and cheap. The PDF folder is found automatically, whether Jupyter is started from the project root or from `notebooks/`.

In [3]:
load_dotenv()  # reads GOOGLE_API_KEY and GROQ_API_KEY from .env

embedding_model = "gemini-embedding-001"
chat_model = "openai/gpt-oss-120b"
docs_dir = next((p for p in [Path("documents"), Path("../documents")] if p.exists()), Path("documents"))
TOP_K = 4

## 3. Load PDFs and split into chunks
Each PDF page is loaded as a document and split into chunks of 1000 characters with a 200-character overlap. Small chunks keep each embedding focused on one idea, and the overlap makes sure a sentence at the edge of one chunk also appears whole in the next. `RecursiveCharacterTextSplitter` tries to cut at paragraphs first, then sentences, then words, so chunks rarely break mid-sentence. These sizes are common defaults that I did not tune.

In [4]:
pdf_paths = sorted(docs_dir.glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError(f"Put at least one PDF in {docs_dir.resolve()}")

docs = [page for p in pdf_paths for page in PyPDFLoader(str(p)).load()]
print(f"Loaded {len(docs)} pages from {len(pdf_paths)} PDF(s)")

chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
for c in chunks:  # drop characters PDFs sometimes produce that crash the embedding API
    c.page_content = c.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

Loaded 16 pages from 1 PDF(s)


## 4. Embeddings, FAISS vector store, retriever, model
Each chunk is turned into a vector with Gemini embeddings, so chunks with similar meaning end up close together. FAISS stores the vectors and finds the ones nearest to a question. I used FAISS because it is free, runs locally and needs no server. The LLM runs on Groq with temperature 0, so answers are as consistent as possible.

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

llm = ChatGroq(model=chat_model, temperature=0, max_retries=8, reasoning_effort="low")

## 5. State and retrieve node
In LangGraph, nodes share information through a state. Here the state holds the question, the retrieved documents and the final answer. The `retrieve` node searches the vector store and saves the top chunks in the state.

In [6]:
class State(TypedDict, total=False):
    question: str
    docs: List[Document]
    answer: str

def retrieve(state: State) -> dict:
    return {"docs": retriever.invoke(state["question"])}

## 6. Generate node
The prompt tells the model to answer only from the retrieved context and to say "I don't know" if the answer is not there. This reduces made-up answers, but it does not fully prevent them.

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only using the provided context. If the answer is not present, say 'I don't know.'"),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])

def generate(state: State) -> dict:
    context = "\n\n".join(d.page_content for d in state["docs"])
    response = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": response.content}

## 7. Build the graph
The graph is a straight line: retrieve, then generate. The printed diagram shows exactly this flow. There is no branching, which is what CRAG adds.

In [8]:
graph = StateGraph(State)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	retrieve --> generate;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 8. Run
I ask a question and print the answer. For a question that my PDF does not cover, the baseline **[answered "I don't know" / answered from unrelated chunks]**. I use the same question in the CRAG notebook so the two can be compared.

In [9]:
def ask(question: str) -> dict:
    result = app.invoke({"question": question})
    print("ANSWER:", result["answer"])
    return result

question = "What does agentic ai means?"
result = ask(question)

ANSWER: I don't know.
